# 8B · Valuing a Company — DCF, Comparables, and the Football Field
### Financial Analytics — Module 8

The investment banker's core craft: **what is this company worth?** You'll answer it for MoneyMart India three independent ways, then combine them the way a real pitch book does:

1. **DCF** — value = the cash the business will generate, discounted to today
2. **Comparables** — value = what the market pays for similar businesses
3. **The football field** — all methods on one chart, as a *range*, never a point

Honesty banner up front: real valuations run 40-tab models with full three-statement forecasts. This notebook is the **skeleton** — every real model is this skeleton wearing more clothes. Learn the skeleton and the clothes make sense forever.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

BASE = "data/"
fin = pd.read_csv(BASE + "company_financials.csv")
latest = fin.iloc[-1]
print(f"MoneyMart FY25-26:  Revenue Rs {latest['revenue_cr']:,.0f} cr | EBITDA Rs {latest['ebitda_cr']:,.0f} cr | PAT Rs {latest['pat_cr']:,.0f} cr")

---
## Part 1 — DCF: the value of future cash, today

### Step 1: Free Cash Flow, the simple honest way

Profit is an opinion; **cash is a fact.** Free Cash Flow (FCF) is the cash the business throws off after reinvesting to keep growing. The full formula has five adjustments; our teaching proxy keeps the two that matter most at this scale:

> **FCF ≈ EBIT × (1 − tax rate) + depreciation − reinvestment**

where reinvestment (capex + working capital) is modelled as a % of revenue — a standard simplification, and we'll test its sensitivity like every other assumption.

In [ ]:
TAX = 0.252                    # from the data: tax/pbt ~ 25.2%
REINVEST_PCT = 0.045           # capex + working capital, as % of revenue (assumption!)

ebit0 = latest["ebit_cr"]; dep0 = latest["depreciation_cr"]; rev0 = latest["revenue_cr"]
fcf0 = ebit0*(1-TAX) + dep0 - REINVEST_PCT*rev0
print(f"Base-year FCF ≈ {ebit0:,.0f}x(1-{TAX}) + {dep0:,.0f} - {REINVEST_PCT:.1%}x{rev0:,.0f} = Rs {fcf0:,.0f} cr")

### Step 2: Project five years, then a terminal value

A DCF has two acts: an **explicit forecast** (5 years, where you claim to know something) and a **terminal value** (everything after, where you claim only that the company matures into steady growth). Assumptions on the table, where they belong:

In [ ]:
GROWTH_Y1_5 = 0.12         # revenue/FCF growth in explicit years (Module 6 taught you: post-COVID trend, stated regime)
TERMINAL_G  = 0.05         # perpetual growth after year 5 - must be <= long-run nominal GDP growth or you've claimed the company eats the economy
WACC        = 0.115        # discount rate: the return investors demand. For an Indian mid-cap retailer, ~11-12%

years = np.arange(1, 6)
fcf = fcf0 * (1 + GROWTH_Y1_5)**years
pv_fcf = fcf / (1 + WACC)**years

# Terminal value at end of year 5 (Gordon growth), then discount THAT back too
tv = fcf[-1]*(1 + TERMINAL_G) / (WACC - TERMINAL_G)
pv_tv = tv / (1 + WACC)**5

ev_dcf = pv_fcf.sum() + pv_tv
print(pd.DataFrame({"year": years, "FCF": fcf.round(0), "PV of FCF": pv_fcf.round(0)}).to_string(index=False))
print(f"\nTerminal value (yr-5 money): Rs {tv:,.0f} cr -> PV Rs {pv_tv:,.0f} cr")
print(f"ENTERPRISE VALUE (DCF): Rs {ev_dcf:,.0f} cr")
print(f"\nShare of value sitting in the terminal: {pv_tv/ev_dcf:.0%}  <- read the next cell before trusting anything")

**That last number is the DCF's dirty secret.** Typically 60–75% of a DCF's value lives in the terminal value — i.e., in the two assumptions (terminal growth, WACC) about which you know *least*. A DCF is less a calculation than **an argument wearing arithmetic**. Which is why no banker ever presents one number:

In [ ]:
# Sensitivity grid: EV across WACC x terminal growth - THE table on every pitch page
waccs = [0.105, 0.11, 0.115, 0.12, 0.125]
gs    = [0.04, 0.045, 0.05, 0.055, 0.06]

def dcf_ev(wacc, g):
    pv = (fcf / (1+wacc)**years).sum()
    return pv + fcf[-1]*(1+g)/(wacc-g)/(1+wacc)**5

grid = pd.DataFrame([[dcf_ev(w, g) for g in gs] for w in waccs],
                    index=[f"WACC {w:.1%}" for w in waccs],
                    columns=[f"g {g:.1%}" for g in gs]).round(0)
print(grid.to_string())
dcf_low, dcf_high = grid.values.min(), grid.values.max()
print(f"\nDCF range: Rs {dcf_low:,.0f} - {dcf_high:,.0f} cr  (half-point wiggles in two assumptions -> {dcf_high/dcf_low-1:.0%} spread!)")

### ✏️ Exercise 1
Where in the grid does the DCF break entirely? Set terminal growth to 6% and WACC to 6.5%. Explain *from the Gordon formula* why the number explodes as g approaches WACC — and what that mathematical cliff implies about trusting any DCF where the two are close.

---
## Part 2 — Comparables: what the market pays for lookalikes

The market prices retail companies every day. Comparables borrow that verdict: find similar listed companies, compute what multiple of earnings/EBITDA they trade at, apply the multiple to MoneyMart. (Peer set below is illustrative — in real work, *choosing the peers is the analysis*: same sector, similar growth, similar margins, similar size. A peer table with a hidden mismatch is a rigged verdict.)

In [ ]:
peers = pd.DataFrame({
    "peer":        ["RetailKing", "ValueBazaar", "UrbanMart", "FreshHyper", "StyleStores"],
    "ev_ebitda":   [14.5, 11.8, 16.2, 12.9, 13.6],
    "pe":          [28.0, 22.5, 34.0, 25.5, 26.8],
    "rev_growth":  [0.14, 0.09, 0.18, 0.11, 0.12],
})
print(peers.to_string(index=False))
print(f"\nEV/EBITDA: median {peers.ev_ebitda.median():.1f}x  (range {peers.ev_ebitda.min():.1f}-{peers.ev_ebitda.max():.1f}x)")
print(f"P/E      : median {peers.pe.median():.1f}x  (range {peers.pe.min():.1f}-{peers.pe.max():.1f}x)")
print("\nNote WHO trades high: UrbanMart, the fastest grower. Multiples are compressed DCFs -")
print("growth and quality live inside them. Never apply a fast-grower's multiple to a slow grower.")

In [ ]:
# Apply peer multiples to MoneyMart's actuals
ebitda, pat = latest["ebitda_cr"], latest["pat_cr"]

ev_ebitda_low, ev_ebitda_high = peers.ev_ebitda.quantile(.25)*ebitda, peers.ev_ebitda.quantile(.75)*ebitda
# P/E gives EQUITY value; convert to enterprise value by adding net debt (assume Rs 600 cr for the toy)
NET_DEBT = 600
pe_low, pe_high = peers.pe.quantile(.25)*pat + NET_DEBT, peers.pe.quantile(.75)*pat + NET_DEBT

print(f"EV/EBITDA method: Rs {ev_ebitda_low:,.0f} - {ev_ebitda_high:,.0f} cr")
print(f"P/E method (EV) : Rs {pe_low:,.0f} - {pe_high:,.0f} cr")
print("\n(Interquartile range, not min-max: outlier peers shouldn't set your goalposts.)")

### ✏️ Exercise 2
MoneyMart grew ~15.5% pre-COVID and ~12% recently. Regress the peers' EV/EBITDA on their revenue growth (scipy linregress, 5 points — a sketch, not science). What multiple does the fitted line imply *at MoneyMart's growth*? This is a "regression comps" mini-version of what banks actually do to justify where in the peer range a target belongs.

---
## Part 3 — The Football Field: the pitch-book chart

In [ ]:
methods = [
    ("DCF (sensitivity range)",      dcf_low,        dcf_high),
    ("EV/EBITDA comps (IQR)",        ev_ebitda_low,  ev_ebitda_high),
    ("P/E comps (IQR, +net debt)",   pe_low,         pe_high),
]

fig, ax = plt.subplots(figsize=(10, 3.6))
for i, (name, lo, hi) in enumerate(methods):
    ax.barh(i, hi-lo, left=lo, height=0.5, color=["#2563EB","#0D9488","#7C3AED"][i], alpha=0.85)
    ax.text(lo-150, i, f"{lo:,.0f}", ha="right", va="center", fontsize=9)
    ax.text(hi+150, i, f"{hi:,.0f}", ha="left",  va="center", fontsize=9)

# The zone where methods AGREE is where conviction lives
overlap_lo = max(m[1] for m in methods); overlap_hi = min(m[2] for m in methods)
if overlap_lo < overlap_hi:
    ax.axvspan(overlap_lo, overlap_hi, color="#F59E0B", alpha=0.18)
    ax.text((overlap_lo+overlap_hi)/2, 2.6, f"convergence zone\nRs {overlap_lo:,.0f}-{overlap_hi:,.0f} cr",
            ha="center", fontsize=9, color="#B45309")

ax.set_yticks(range(len(methods)), [m[0] for m in methods])
ax.set_title("MoneyMart India - Enterprise Value football field (Rs crore)", loc="left", fontweight="bold")
ax.set_xlabel("Rs crore")
plt.tight_layout(); plt.show()

print(f"The pitch sentence: 'Methods triangulate to roughly Rs {overlap_lo:,.0f}-{overlap_hi:,.0f} cr;")
print("we'd anchor negotiations in this zone, with the DCF grid as the walk-away logic.'")

**Why bankers chart ranges, not points:** each method has different blind spots — DCF trusts your forecasts, comps trust the market's mood and your peer choice. Where independent methods *overlap* is where conviction is earned. A single-number valuation is Module 6's "guess wearing a suit," in a more expensive suit.

### ✏️ Exercise 3
Recession scenario: peers de-rate 20% (all multiples ×0.8) and your DCF growth drops to 8%. Rebuild the football field. Does a convergence zone survive? What does its shrinkage (or disappearance) tell a deal team about *timing*?

---
## The cross-section lesson, in one paragraph

Look at what you used: **Module 6's** trend/regression (growth assumptions, regression comps), **Module 7's** sensitivity discipline (the WACC×g grid is a tornado in table form), **Module 4's** exhibits (the field IS a ranking-with-ranges chart), **Module 1's** skepticism (point-in-time earnings, peer-selection bias). Investment banking has no private mathematics — it is the *cross-section* where every stream's tools meet a deal. That's Module 8's whole thesis, demonstrated in one notebook.

*AI disclosure: ______*

In [ ]:
# workspace
